# Sequence-to-Sequence English-to-Hindi Translation

Implementation of an LSTM-based encoder–decoder model for translating short English sentences into Hindi.

**Environment:** Google Colab / Jupyter Notebook  
**Language:** Python

## 1. Aim
Build a basic sequence-to-sequence (Seq2Seq) encoder–decoder model that learns to translate short English sentences into Hindi.

## 2. Theory
A Seq2Seq network maps an input sequence to an output sequence. The **encoder** reads the source sentence and produces a context representation. The **decoder** uses that representation and previously generated target tokens to predict the next target token.

This practical uses an LSTM encoder and LSTM decoder with teacher forcing. A small illustrative parallel corpus is included so the notebook runs without downloading a dataset. It demonstrates the complete pipeline, but it is not a production-quality translator. Better translation quality requires a much larger, carefully aligned English–Hindi corpus, subword tokenization, and model tuning.

## 3. Import libraries

In [ ]:
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow version:", tf.__version__)

## 4. Prepare a small parallel English–Hindi dataset
Each English sentence is paired with its Hindi translation. `<start>` and `<end>` mark target sequence boundaries.

In [ ]:
pairs = [
    ("hello", "नमस्ते"),
    ("good morning", "सुप्रभात"),
    ("how are you", "आप कैसे हैं"),
    ("i am fine", "मैं ठीक हूँ"),
    ("thank you", "धन्यवाद"),
    ("what is your name", "आपका नाम क्या है"),
    ("my name is ram", "मेरा नाम राम है"),
    ("i love india", "मुझे भारत से प्यार है"),
    ("good night", "शुभ रात्रि"),
    ("please help me", "कृपया मेरी मदद करें"),
    ("where are you going", "आप कहाँ जा रहे हैं"),
    ("i am a student", "मैं एक विद्यार्थी हूँ"),
    ("this is a book", "यह एक किताब है"),
    ("open the door", "दरवाजा खोलो"),
    ("close the window", "खिड़की बंद करो"),
    ("the weather is good", "मौसम अच्छा है"),
    ("i want water", "मुझे पानी चाहिए"),
    ("come here", "यहाँ आओ"),
    ("see you tomorrow", "कल मिलते हैं"),
    ("welcome home", "घर में आपका स्वागत है"),
]

english_texts = [src.lower().strip() for src, _ in pairs]
hindi_texts = ["<start> " + tgt.strip() + " <end>" for _, tgt in pairs]

print("Number of sentence pairs:", len(pairs))
for src, tgt in pairs[:5]:
    print(f"{src:25s} -> {tgt}")

## 5. Tokenize sentences
For this compact demonstration, whitespace tokenization is used. A production system should use a subword tokenizer such as SentencePiece or WordPiece.

In [ ]:
def tokenize(text):
    return text.split()

source_tokens = sorted({tok for sentence in english_texts for tok in tokenize(sentence)})
target_tokens = sorted({tok for sentence in hindi_texts for tok in tokenize(sentence)})

source_word_to_id = {"<pad>": 0, "<unk>": 1}
for token in source_tokens:
    source_word_to_id[token] = len(source_word_to_id)

target_word_to_id = {"<pad>": 0, "<unk>": 1}
for token in target_tokens:
    target_word_to_id[token] = len(target_word_to_id)

source_id_to_word = {i: w for w, i in source_word_to_id.items()}
target_id_to_word = {i: w for w, i in target_word_to_id.items()}

source_vocab_size = len(source_word_to_id)
target_vocab_size = len(target_word_to_id)
max_encoder_len = max(len(tokenize(s)) for s in english_texts)
max_decoder_len = max(len(tokenize(s)) for s in hindi_texts)

print("Source vocabulary:", source_vocab_size)
print("Target vocabulary:", target_vocab_size)
print("Maximum source length:", max_encoder_len)
print("Maximum target length:", max_decoder_len)

## 6. Convert text into padded sequences
The decoder input is the target sentence without its final token. The decoder target is the same sequence shifted one position to the left.

In [ ]:
def encode_sentence(sentence, word_to_id, max_len):
    ids = [word_to_id.get(tok, word_to_id["<unk>"]) for tok in tokenize(sentence)]
    return ids + [0] * (max_len - len(ids))

encoder_input_data = np.array([
    encode_sentence(s, source_word_to_id, max_encoder_len)
    for s in english_texts
], dtype="int32")

target_full = np.array([
    encode_sentence(s, target_word_to_id, max_decoder_len)
    for s in hindi_texts
], dtype="int32")

decoder_input_data = target_full[:, :-1]
decoder_target_data = target_full[:, 1:, np.newaxis]

# Ignore padding positions when calculating the loss.
sample_weights = (decoder_target_data[:, :, 0] != 0).astype("float32")

print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target shape:", decoder_target_data.shape)

## 7. Build the encoder–decoder model
The encoder's final hidden and cell states initialise the decoder. The decoder predicts a probability distribution over the Hindi vocabulary at every time step.

In [ ]:
embedding_dim = 64
latent_dim = 128

# Encoder
encoder_inputs = Input(shape=(None,), name="encoder_inputs")
encoder_embedding = Embedding(source_vocab_size, embedding_dim, mask_zero=True,
                              name="encoder_embedding")(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True, name="encoder_lstm")
_, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# Decoder (training model)
decoder_inputs = Input(shape=(None,), name="decoder_inputs")
decoder_embedding_layer = Embedding(target_vocab_size, embedding_dim, mask_zero=True,
                                    name="decoder_embedding")
decoder_embedded = decoder_embedding_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True,
                    name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_embedded, initial_state=encoder_states)
decoder_dense = Dense(target_vocab_size, activation="softmax", name="output_softmax")
decoder_outputs = decoder_dense(decoder_outputs)

training_model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
training_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                       metrics=["accuracy"])
training_model.summary()

## 8. Train the model
Because the corpus is intentionally tiny, the model is trained to demonstrate the workflow and may memorise these examples. Training accuracy on this dataset should not be interpreted as general translation performance.

In [ ]:
history = training_model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    sample_weight=sample_weights,
    batch_size=4,
    epochs=250,
    validation_split=0.1,
    verbose=0
)
print("Training completed.")
print("Final training loss:", round(float(history.history["loss"][-1]), 4))
print("Final training accuracy:", round(float(history.history["accuracy"][-1]), 4))

## 9. Build inference models
During inference, the decoder predicts one token at a time. The predicted token and updated LSTM states are fed into the next step.

In [ ]:
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,), name="decoder_state_h")
decoder_state_input_c = Input(shape=(latent_dim,), name="decoder_state_c")
decoder_single_input = Input(shape=(1,), name="decoder_token_input")

decoder_single_embedded = decoder_embedding_layer(decoder_single_input)
decoder_single_output, next_h, next_c = decoder_lstm(
    decoder_single_embedded,
    initial_state=[decoder_state_input_h, decoder_state_input_c]
)
decoder_single_output = decoder_dense(decoder_single_output)

decoder_model = Model(
    [decoder_single_input, decoder_state_input_h, decoder_state_input_c],
    [decoder_single_output, next_h, next_c]
)

def translate(sentence, max_steps=20):
    sentence = sentence.lower().strip()
    input_ids = [source_word_to_id.get(tok, source_word_to_id["<unk>"])
                 for tok in tokenize(sentence)]
    input_ids = input_ids[:max_encoder_len]
    input_ids += [0] * (max_encoder_len - len(input_ids))
    states = encoder_model.predict(np.array([input_ids]), verbose=0)

    start_id = target_word_to_id["<start>"]
    end_id = target_word_to_id["<end>"]
    token_id = start_id
    output_words = []

    for _ in range(max_steps):
        probs, h, c = decoder_model.predict(
            [np.array([[token_id]]), states[0], states[1]], verbose=0
        )
        token_id = int(np.argmax(probs[0, -1, :]))
        if token_id == end_id or token_id == 0:
            break
        output_words.append(target_id_to_word.get(token_id, "<unk>"))
        states = [h, c]

    return " ".join(output_words)

print("Example translation:")
print("English: hello")
print("Hindi:  ", translate("hello"))

## 10. Test on sample sentences

In [ ]:
test_sentences = ["hello", "thank you", "good morning", "i love india",
                  "please help me", "where are you going"]

for sentence in test_sentences:
    print(f"English: {sentence}")
    print(f"Hindi:   {translate(sentence)}")
    print("-" * 40)

## 11. Observations
- The encoder converts the English input sequence into learned internal states.
- The decoder generates the Hindi output sequentially.
- Teacher forcing supplies the correct previous target token during training.
- A small dataset is useful for demonstrating architecture, but it cannot teach broad grammar or vocabulary.

## 12. Limitations and improvements
1. Expand the aligned bilingual dataset substantially.
2. Use subword tokenization to handle unseen words and morphology.
3. Use train/validation/test splits and BLEU or chrF for evaluation.
4. Try attention or Transformer-based encoder–decoder architectures.
5. Save the tokenizer vocabularies and model weights together.

## 13. Conclusion
A basic LSTM-based Seq2Seq encoder–decoder for English-to-Hindi translation was implemented. The notebook demonstrates data preparation, tokenisation, teacher-forced training, inference, and sample translation.